# vox-ar — فيديو Vox عربي بستايل الجرايد (مجاناً، بدون تثبيت)

**الخطوات:** 
1. ارفع `vox-ar.zip` إلى Google Drive (المجلد الرئيسي) — أو اتركه واستخدم الرفع المباشر في الخلية 2.
2. اضغط على أيقونة المفتاح 🔑 يسار الشاشة → أضف سرّاً باسم `GEMINI_API_KEY` وفعّل "Notebook access". (مجاني من https://aistudio.google.com/apikey)
3. من القائمة: **Runtime → Run all**. الفيديو النهائي يُحفظ في Drive: `vox-ar-output/<project>/final.mp4`.

المعاينة (بوابة الشكل) تظهر بعد الخلية 4 — لو ما عجبتك، عدّل `beats.json` وأعد التشغيل من الخلية 5.

In [ ]:
#@title 1) الإعداد — ربط Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2) جلب السكيل من Drive (أو رفعه مباشرة) + تثبيت المكتبات
import os, glob, zipfile
zips = glob.glob('/content/drive/MyDrive/vox-ar.zip') + glob.glob('/content/drive/MyDrive/**/vox-ar.zip', recursive=True)
if not zips:
    from google.colab import files
    up = files.upload()          # اختر vox-ar.zip من جهازك
    zips = [k for k in up if k.endswith('.zip')]
zipfile.ZipFile(zips[0]).extractall('/content')
os.chdir('/content/vox-ar')
!pip -q install -r requirements.txt 2>&1 | tail -1
print('ready:', os.getcwd(), '|', zips[0])

In [ ]:
#@title 3) اختر المشروع والإعدادات
PROJECT = "examples/ai-history"   #@param ["examples/ai-history", "examples/screen-time"] {allow-input: true}
PROVIDER = "gemini"               #@param ["gemini", "pollinations", "placeholder"]
PLATES = "procedural"             #@param ["procedural", "ai"]
TTS = "edge"                      #@param ["edge", "gemini", "placeholder"]
PERSONA = "documentary_male"      #@param ["documentary_male", "documentary_female", "levant_male", "levant_female", "gulf_male", "energetic", "calm_teacher", "gemini_narrator", "gemini_warm", "gemini_energetic"]
MOOD = "investigative"            #@param ["investigative", "tense", "hopeful", "upbeat", "epic", "calm", "tech", "kids"]
import os, json, shutil
from google.colab import userdata
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception as e:
    print('⚠️ أضف GEMINI_API_KEY في أسرار Colab (🔑) إذا اخترت provider=gemini')
name = os.path.basename(PROJECT)
os.makedirs('out', exist_ok=True)
if not os.path.exists(f'out/{name}'):
    shutil.copytree(PROJECT, f'out/{name}')
p = f'out/{name}/beats.json'; d = json.load(open(p, encoding='utf-8'))
d['provider'] = PROVIDER; d['plates'] = PLATES
d['tts'] = {'engine': TTS, 'persona': PERSONA}
d['music'] = dict(d.get('music', {}), path='auto', mood=MOOD)
json.dump(d, open(p, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('project:', name, '| provider:', PROVIDER, '| plates:', PLATES, '| persona:', PERSONA, '| mood:', MOOD)

In [ ]:
#@title 4) الصور + معاينة اللقطات (بوابة الشكل)
!python run.py out/{name} --gate
from IPython.display import Image, display
display(Image(f'out/{name}/preview/contact_sheet.png'))

In [ ]:
#@title 5) الصوت + المونتاج + التجميع → final.mp4  (وحفظ في Drive)
!python run.py out/{name} --workers 2
import shutil, os
dst = f'/content/drive/MyDrive/vox-ar-output/{name}'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'out/{name}/final.mp4', dst)
shutil.copy(f'out/{name}/beats.json', dst)
print('✅ saved →', dst)
from IPython.display import HTML
from base64 import b64encode
mp4 = open(f'out/{name}/final.mp4','rb').read()
HTML(f'<video width=720 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')